In [ ]:
# Dynamic Time Warping (DTW) for Speech
#
# This program demonstrates how to use DTW to compare two speech samples.
#
# It does *not* run DTW on the raw waveform. Instead, it uses a standard
# speech processing technique:
#
# 1. Load two audio files.
# 2. Extract Mel-Frequency Cepstral Coefficients (MFCCs) from each.
#    MFCCs are a feature that represents the "timbre" or "quality" of a
#    sound, which is much more robust for comparison than raw amplitude.
# 3. Run DTW on the *sequence of MFCCs* to find the optimal alignment.
# 4. Visualize the waveforms, their MFCCs, and the final alignment path.
#
# To run this code, you will need to install:
# pip install librosa matplotlib

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np


In [ ]:

def run_speech_dtw_analysis():
    """
    Main function to load audio, process MFCCs, run DTW, and plot results.
    """
    
    print("Loading audio samples...")
    # 1. Load Audio
    # We use librosa's built-in example audio files (from the LibriSpeech dataset)
    # so you can run this code without providing your own files.
    # We load at a consistent sample rate (sr) for comparison.
    SR = 22050
    y1, sr1 = librosa.load(librosa.example('libri1'), sr=SR)
    y2, sr2 = librosa.load(librosa.example('libri2'), sr=SR)

    print("Extracting MFCCs...")
    # 2. Extract MFCCs
    # We use 13 MFCCs, which is a standard number for this task.
    # This turns each audio file into a 2D array: (13 MFCCs x N frames)
    mfcc1 = librosa.feature.mfcc(y=y1, sr=sr1, n_mfcc=13)
    mfcc2 = librosa.feature.mfcc(y=y2, sr=sr2, n_mfcc=13)
    
    # mfcc1.shape will be (13, N), where N is the number of frames
    # mfcc2.shape will be (13, M), where M is the number of frames
    
    print("Running Dynamic Time Warping...")
    # 3. Run DTW
    # librosa.dtw finds the optimal path between two 2D feature arrays.
    # 'metric' specifies how to compare two MFCC frames (vectors):
    # 'euclidean' is a standard distance measure.
    #
    # D is the accumulated cost matrix.
    # wp is the warping path (an array of index pairs).
    D, wp = librosa.dtw(X=mfcc1, Y=mfcc2, metric='euclidean')

    # The warping path is returned from end-to-start, so we flip it
    wp = np.flip(wp, axis=0)

    # Get the total cost of the alignment
    total_cost = D[-1, -1]
    
    print(f"\n--- DTW Analysis Complete ---")
    print(f"Audio 1 (libri1) frames: {mfcc1.shape[1]}")
    print(f"Audio 2 (libri2) frames: {mfcc2.shape[1]}")
    print(f"Total DTW Cost: {total_cost:.4f}")
    print("Lower cost means higher similarity between the speech samples.")
    print("Displaying visualizations...")

    # 4. Plot the results
    fig = plt.figure(figsize=(12, 12))
    
    # --- Plot 1: Waveform 1 ---
    ax1 = fig.add_subplot(3, 2, 1)
    librosa.display.waveshow(y1, sr=sr1, ax=ax1, color='blue', alpha=0.7)
    ax1.set_title('Waveform 1 (libri1)')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Amplitude')

    # --- Plot 2: Waveform 2 ---
    ax2 = fig.add_subplot(3, 2, 2)
    librosa.display.waveshow(y2, sr=sr2, ax=ax2, color='green', alpha=0.7)
    ax2.set_title('Waveform 2 (libri2)')
    ax2.set_xlabel('Time (s)')
    
    # --- Plot 3: MFCC 1 ---
    ax3 = fig.add_subplot(3, 2, 3)
    librosa.display.specshow(mfcc1, sr=sr1, x_axis='time', ax=ax3)
    ax3.set_title('MFCC 1')
    ax3.set_ylabel('MFCC Coefficient')

    # --- Plot 4: MFCC 2 ---
    ax4 = fig.add_subplot(3, 2, 4)
    librosa.display.specshow(mfcc2, sr=sr2, x_axis='time', ax=ax4)
    ax4.set_title('MFCC 2')
    
    # --- Plot 5: DTW Alignment ---
    # This is the main "result" plot.
    # It shows the cost matrix 'D' as a heatmap and plots the
    # optimal warping path 'wp' on top.
    ax5 = fig.add_subplot(3, 1, 3)
    # We'll display the accumulated cost matrix D
    img = librosa.display.specshow(D, sr=sr1, x_axis='frames', y_axis='frames', ax=ax5, cmap='gray_r')
    
    # Plot the warping path
    ax5.plot(wp[:, 1], wp[:, 0], label='Warping Path', color='red', linewidth=3)
    
    ax5.set_title('DTW Cost Matrix & Alignment Path')
    ax5.set_xlabel('Audio 2 Frames (MFCC)')
    ax5.set_ylabel('Audio 1 Frames (MFCC)')
    ax5.legend()
    
    # Add a colorbar to show the cost
    fig.colorbar(img, ax=ax5, label='Accumulated Cost')
    
    plt.tight_layout()
    plt.show()

if __name__ == '__main__':
    run_speech_dtw_analysis()